# 121 — Real flow-step PCP pilot for SmolVLA

This worker applies the correction **inside the actual flow sampler**, using the P&P probe's clean estimate, the frozen scalar Q10 input gradient, and re-noising at the same noise level with the same probe epsilon. The sampler then completes its remaining Euler steps. No completed action chunk is edited.

Default correction: zero-based Euler step 3 of ten, `s=0.7`, the existing `K=1` P&P probe. Steps 1/2 retain their K=3/1 P&P. You can choose step 2 (`s=0.8`, also K=1). K=1 makes the mean and last clean estimates identical. Zero correction must reproduce the unmodified P&P **entire final chunk bit for bit** and consume the same perturbation RNG; otherwise generation stops.

The matched controls use identical source initial noise and cloned source probe streams: zero correction, positive Q gradient, negative Q gradient, and independent random directions matched in clean-action RMS at .02/.06. Only the first ten seven-dimensional actions enter Q; frozen cached prefix, robot state, proprioception and known preaction masks are held constant. Clean tail/padded dimensions are preserved and all flow dimensions retain the same epsilon. Full denoising can propagate a prefix correction into the tail.

`CORRECTION_MODE='rms'` uses a bounded normalized-gradient PCP update with effective lambda = radius / gradient RMS. `'lambda'` applies the raw lambda times Q-gradient (the same .02/.06 coefficients), with random controls matched to its realized RMS. The default clean RMS .02/.06 causes immediate latent RMS .006/.018 at s=.7; final action displacement is measured separately.

The critic was trained on terminal action proposals; using it on intermediate clean estimates is **extrapolation**. This is an exploratory test of the real PCP mechanism using that critic, not a claim that it improves policy. These 18 roots overlap training/validation and retain their original split. The historical stored P&P source is kept alongside regenerated zero PCP and a fresh P&P candidate.

The worker first benchmarks the critic against existing same-state candidate outcomes on CPU. Simulator generation is disabled by `RUN_INTERVENTION=False`. Use three copies with SHARD_INDEX=0,1,2 and ROOT_LIMIT=1 for smoke. Supply a scalar checkpoint manually by upload or Drive path; no checkpoint is fetched from or uploaded to Supabase.


In [ ]:
SHARD_INDEX = 0  # 0, 1, or 2
ROOT_LIMIT = 1  # smoke default; None means all six roots in this shard
RUN_INTERVENTION = False
CHECKPOINT_PATH = ''  # /content/uploaded.pt or /content/drive/MyDrive/.../checkpoint.pt
UPLOAD_CHECKPOINT = False  # Optional forced upload; an empty path also prompts automatically
MOUNT_DRIVE = False  # True for a Drive checkpoint
DEVICE = 'cpu'  # CPU is sufficient for the first benchmark
CORRECTION_STEP = 3  # existing zero-based K=1 hook: 3 gives s=.7; 2 gives s=.8
CORRECTION_MODE = 'rms'  # 'lambda' for raw lambda * probability-gradient


In [ ]:
# Install NVIDIA EGL only when simulator intervention is enabled, before MuJoCo imports.
if RUN_INTERVENTION:
    import re, subprocess
    driver = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        text=True).splitlines()[0].strip()
    package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    availability = subprocess.check_output(['apt-cache', 'policy', package], text=True)
    if 'Candidate: (none)' in availability:
        raise RuntimeError(f'{package} is unavailable in this runtime')
    subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', package], check=True)
    subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if UPLOAD_CHECKPOINT or not CHECKPOINT_PATH:
    from google.colab import files
    from pathlib import Path
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one scalar critic .pt checkpoint.')
    CHECKPOINT_PATH = str(Path(next(iter(uploaded))).resolve())

EXTRAS = 'sim'
SETUP_ENV = RUN_INTERVENTION
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
from pnp.smolvla_flow_pcp_pilot import run_flow_pcp_pilot
if not CHECKPOINT_PATH:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one scalar critic .pt checkpoint.')
    CHECKPOINT_PATH = str(Path(next(iter(uploaded))).resolve())
if not Path(CHECKPOINT_PATH).is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}. Supply a Colab/Drive path or clear CHECKPOINT_PATH to upload.')
assert SHARD_INDEX in (0, 1, 2)
REPORT_PATH = f'/content/flow_pcp_pilot_shard_{SHARD_INDEX}.json'
report = run_flow_pcp_pilot(
    correction_step=CORRECTION_STEP, correction_mode=CORRECTION_MODE,
    checkpoint_path=CHECKPOINT_PATH, shard_index=SHARD_INDEX,
    root_limit=ROOT_LIMIT, run_intervention=RUN_INTERVENTION,
    device=DEVICE, output_path=REPORT_PATH)
print({'report': REPORT_PATH, 'new_simulator_roots': report['new_roots']})


After reviewing the CPU gradient benchmark, opt in by setting RUN_INTERVENTION=True and selecting a CUDA runtime. Rerun setup so NVIDIA EGL and LIBERO are available. Every generated control is injected into the same restored root and gets the frozen P&P future seed. A new `flow-step-pcp` experiment stores full branch artifacts, critic SHA/architecture, correction step/s, probe-noise digests, clean/latent/final displacement and paired outcomes. Complete groups resume safely.

Inspect zero PCP versus the historical source, ±gradient versus zero PCP, and random versus zero PCP at each strength. Historical decode disagreement is recorded; the causal baseline is regenerated zero PCP. These overlapping roots and weak/extrapolated gradients do not establish independent policy performance.
